### **tensor prep**

# CNN Tensor-Building Notebook

The purpose of this notebook is to create tensors for use in a CNN designed to identify clear and robust markers of climate change.

### Imports and File Stitching

In [ ]:
# Imports
import os
import re
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt

# ----File Stitching----
# If in prep folder, cd back to base repository folder
if os.path.basename(os.getcwd()) != "MamalakisResearch":
    os.chdir('../..')

# Control which user's files are accessed
match_sk = 'sophiekim'
match_hc = 'hayeonchung'
match_ck = 'Caroline'
match_st = 'student'
path_str = os.getcwd()
print(path_str)
if re.search(match_sk, path_str):
    os.chdir("/Users/sophiekim/Desktop/2_research/MamalakisResearch") 
    user = match_sk
    print("Sophie Kim recognized as user.")
elif re.search(match_hc, path_str):
    os.chdir("/Users/hayeonchung/Downloads/Mamalakis Graduate Research/MamalakisResearch") 
    user = match_hc
    print("Hayeon Chung recognized as user.")
elif re.search(match_st, path_str):
    os.chdir("/Users/student/Desktop/mamalakis_research/MamalakisResearch") # Caroline's computer
    user = "carolinekranefuss"
    print("Caroline Kranefuss recognized as user.")
elif re.search(match_ck, path_str):
    os.chdir("/Users/Caroline/Desktop/school/MamalakisResearch") 
    user = "carolinekranefuss2"
    print("Caroline Kranefuss 2 recognized as user.")
else:
    print("User not recognized. Please manually change to parent repository directory 'MamalakisResearch'.")

In [ ]:
# Assign base path
base_path = os.getcwd()

# All users should have locally loaded 'data' folder
data_path = base_path + '/data/'

In [ ]:
# initializing model list variable to call in function
model_list = [
    "CNRM_ESM2-1_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MIROC6_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MPI-ESM1-2-LR_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MRI-ESM2-0_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "UKESM1-0-LL_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
]

In [ ]:
# pulling variables out for the plot function 
VAR_LIST = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind"]

# dictionary for each unit for the plot 
UNIT_MAP = {
    "tas": "°C", 
    "tasmax": "°C", 
    "tasmin": "°C", 
    "pr": "mm/day", 
    "psl": "hPa", 
    "sfcWind": "m/s"}

## **Functions**

### **converting units**

*converting units for the variables*

In [ ]:
def convert_units(varname: str, x: np.ndarray):
    """
    varname: index number from the list of variables so get_data func can convert units 
    x: data that needs units changed (raw x data) in get_data func 
    """
    if varname in {"tas", "tasmax", "tasmin"}:
        # kelvin to celsius
        return x - 273.15, "$^{\circ}$C"
    if varname == "pr":
        # kg/(m2*s) to mm/day
            # 1kg/m2 = 1 mm 
        return x * 86400.0, "mm/day"
    if varname == "psl":
        # pascals to hpa
        return x / 100.0, "hPa"
    
    # this doesn't need to be converted -- just adding the units 
    if varname == "sfcWind":
        return x, "m/s"
    
    return x, "unknown"

### **cnn tensor prep**

*calculates x_data tensor that calculates statistical summary for a certain variable (ie tas, tasmax, etc.) for specified early and late periods (also a baseline). can also calculate the difference between the early and late periods with the baseline yrs of 2015 and 2024. returns x_data tensor and y_data tensor (y_data tensor gives back a 2d tensor of [# of samples, 1] that gives binary labels (0 for early, 1 for late) for every variable value calculated)*

In [ ]:
def get_model_name(path: str) -> str:
    """
    helper function so that get_cnn_tensors prints out the models that are being processed
    pulled from hayeon's code 
    """
    # Everything before "_ssp..."
    return os.path.basename(path).split("_ssp")[0]

In [ ]:
def get_cnn_tensors(model_list, scenario, data_path, 
                    st_early=2015, end_early=2024, 
                    st_late=2055, end_late=2064,
                    stat='mean', use_anomaly=True, 
                    models_to_run=None, file_start_year=2015, 
                    vars_to_use=None):
    
    """
    model_list: variable of list of models
    scenario: input as either 'ssp119' or 'ssp126' strings
    data_path: variable of data_path for user
    st_early: early period start yr integer
    end_early: early period end yr integer
    st_late: late period start yr integer
    end_late: late period end yr integer
    stat: what stat function user wants to run to summarize variables 
        mean: mean
        std: standard deviation
        max: max val
        min: min val
        medium: median
    use_anomaly: whether to subtract values from baseline values to find anomaly
        True: subtract values
        False: don't subtract values 
    models_to_run: can specify number of models to run by index of model_list variable
        None: all models 
        [x]: one model to run 
        [x, x, x...]: whatever number of models to run 
    file_start_year: ensuring that if time dimension in models starts at 0 or 1, the func will slice the time correctly 
    vars_to_use: give list of strings of vars to calculate, if none specified (aka default) then does all 7 vars
    """
    
    # -------------- VARIABLE / STATS / MODELS DEFINITION ----------------

    # defining variable list for unit conversion later 
    var_list = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind"]
    # default of calculating with all 7 vars
    if vars_to_use is None:
        selected_vars = var_list
    # if user specified certains vars to be calculated
    else:
        # check to see that the string items provided do exist
        selected_vars = [v for v in vars_to_use if v in var_list]

    # find the indexes of the specified variables defined by user (or defaulting to finding all the vars' indices)
    var_indices = [var_list.index(v) for v in selected_vars]

    # mapping out stats and the relevant func
    stat_map = {'mean': np.nanmean, 'std': np.nanstd, 'max': np.nanmax, 'min': np.nanmin, 'median': np.nanmedian}
    # variable that will calculate the stat for whatever the user wants and then will default to mean (could also leave open-ended) 
    calc_func = stat_map.get(stat.lower(), np.nanmean)
    # runs all models
    if models_to_run is None:
        selected_models = model_list
    # runs one model if user only specified one (int not list)
    elif isinstance(models_to_run, int):
        selected_models = [model_list[models_to_run]]
    # if none of the above options (all or one) was selected, then was list of models so get those models 
    else:
        selected_models = [model_list[i] for i in models_to_run]

    # initializing lists 
    x_list, y_list = [], []


    # ----------------- BASELINE CALCULATIONS -------------

    # opening file for each model 
    for filename in selected_models:
        full_path = os.path.join(data_path, filename)
        model_short_name = get_model_name(filename)
        print(f"processing model: {model_short_name}")
        
        with nc.Dataset(full_path) as ds:
            # loading the data for specific model 
                # slicing dimensions for the ensembles, the specific var indices and all the times/lon/lat dimensions
            data_all_vars = ds[f"data_{scenario}"][:, var_indices, :, :, :] 

            data_all_vars = data_all_vars.filled(np.nan)

            all_baseline_avgs = []
            all_baseline_stds = []
            # starting unit conversions for ALL variables
            # looping through all 7 vars or the selected vars only 
                # using enumerate to keep track of the index of var and what the var is in the var_list defined above
                    # idx to keep track of what slice of the var dimension 
                    # var_name so that convert_units can be called correctly 
            for idx, var_name in enumerate(selected_vars):
                # converting var to relevant unit from func -- slicing the relevant var one at a time (getting all the info for all the other dimensions, just associated with that certain var)
                    # returns the converted numbers and string that gives converted unit
                converted_data, _ = convert_units(var_name, data_all_vars[:, idx, :, :, :])
                data_all_vars[:, idx, :, :, :] = converted_data
        
                # baseline = first 10 yrs --> for one variable
                baseline_slice = data_all_vars[:, idx, 0:120, :, :] #(5, 120, 144, 73)
                #print(baseline_slice.shape)

                n_models, n_months, n_lat, n_lon = baseline_slice.shape 
                n_years = n_months // 12 # 10 # Reshape to (5, 10, 12, 144, 73), then mean over the months axis 
                yearly_baseline_slices = baseline_slice.reshape(n_models, n_years, 12, n_lat, n_lon).mean(axis=2) # (5, 10, 144, 73)

                # average ensembles (0 dimen) and yrs (1 dimen)
                var_avg = np.nanmean(yearly_baseline_slices, axis=(0,1), dtype = np.float64)  # (144, 73) -- one global map per variable and model 
      
                all_baseline_avgs.append(var_avg)
          

                # ------------- MANUAL STD CALCULATION ----------------
                
                #### population versus sample? 
                    # looping through each model but each model has 5 ensembles
                squared_diffs = (yearly_baseline_slices - var_avg)**2
                # print(squared_diffs.shape)
                var_std = np.sqrt(np.nanmean(squared_diffs, axis=(0,1)), dtype = np.float64)

                # ---------------- QUANTILE THRESHOLD -------------------

                # 1% quantile threshold 
                threshold = np.nanquantile(var_std, 0.01)
                            # make any std variable value that is less than or equal to threshold NAN         
                var_std = np.where(var_std <= threshold, np.nan, var_std) 

                all_baseline_stds.append(var_std)
                            
            var_std_array = np.array(all_baseline_stds)
            
            var_avg_array = np.array(all_baseline_avgs) # (7, 144, 73)

            
            # -------------------- EARLY AND LATE PERIOD CALCULATIONS -------------

            # making list of periods where early period assigned 0 and late assigned 1 
            periods = [(st_early, end_early, 0), (st_late, end_late, 1)]

            # going thru periods 
            for start_yr, end_yr, label in periods:
                # making var for the number of ensembles that are getting looked at 
                n_ens = data_all_vars.shape[0]
                
                # going thru every yr within each period 
                for yr_idx in range(start_yr, end_yr + 1):
                    # indexing by month 
                    m_idx_start = (yr_idx - file_start_year) * 12
                    m_idx_end = m_idx_start + 11 # ??? We think
                    
                    # going thru individual ensemble for current yr 
                    for ens_idx in range(n_ens):
                        # slicing to get the info at the ensemble index, all vars, time index, all lats/longs
                        annual_slice = data_all_vars[ens_idx, :, m_idx_start:m_idx_end+1, :, :]
                        # applying user specified stat func to make the monthly vals into aggregated yearly vals
                        yearly_val = calc_func(annual_slice, axis=1) # (7, 144, 73)
   
                        if use_anomaly:
                            # calculate val from subtracting the value from the above calculated mean and then divide by standard deviation
                            val = (yearly_val - var_avg_array) / var_std_array
                        else:
                            val = yearly_val
                        
                        x_list.append(val)
                        y_list.append(label)


    # ----------------- VARIALBES TO BE RETURNED ---------------------                    
    
    x_list = np.nan_to_num(np.array(x_list), nan=0.0)
    y_list = np.array(y_list).reshape(-1, 1)

    return x_list, y_list, var_std_array

### **plot function**

*this provides a plot calculated from the x_data and y_data tensors from above function. splits x_data into its respective early and late periods using the indices of binary labels from the y_data. values on plot are averages across the values for the early and late periods, besides the third plot which provides the signal (or difference between the early and late periods).*

In [ ]:
def plot_all_variables(x_data, y_data):
    """
    x_data: shape (samples (should be 500), 7, lat, lon)
    y_data: shape (samples (should be 500), 1)
    """
    # initializing variable list 
    var_list = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind"]
    
    # indexing out where the early and late periods are 
        # adding [0] to end so that it doesn't give back the actual y data labels, just returns the row index 
    early_indices = np.where(y_data == 0)[0]
    late_indices = np.where(y_data == 1)[0]
    
    # PLOTTING INDIVID VARS
    # looping through the var list and ensuring that the var_name is also saved when going through each index val in list
    for i, var_name in enumerate(var_list):
        # calculating the avg for all the early period rows and for whatever var value
            # axis 0 collapses sample dimension (first dimension) so you get the average vals for the variable for each lat/lon
        early_mean = np.mean(x_data[early_indices, i, :, :], axis=0)
        late_mean = np.mean(x_data[late_indices, i, :, :], axis=0)
        # using the helper func below to plot for each var 
        plot_comparison_grid(early_mean, late_mean, f"variable: {var_name}")


    # PLOTTING ALL VARS AVERAGED
    # averaging al the x_data tg based on variable dimension 
    multivariate_data = np.mean(x_data, axis=1) 
    
    # indexing out where the early and late periods are 
    early_multi = np.mean(multivariate_data[early_indices], axis=0)
    late_multi = np.mean(multivariate_data[late_indices], axis=0)
    
    # using helper func below to plot this one 
    plot_comparison_grid(early_multi, late_multi, "multivariate plot of all 7 vars averaged together")


def plot_comparison_grid(early_map, late_map, main_title):

    # calculating the diff between the late and early
        # positive = var increased (warmer, wetter, etc)
        # negative = var decreased (drier, colder, etc) 
    signal = late_map - early_map
    # transposing everything so that lat and longs make sense 
    maps = [early_map.T, late_map.T, signal.T]
    # gives titles for the 3 subplots 
    titles = ["early period avg", "late period avg", "difference (late - early period)"]
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    plt.suptitle(main_title, fontsize=18, fontweight='bold', y=1.05)
    
    # loop runs three times for early, late and difference 
    for j in range(3):
        # for the early and late plots 
        if j < 2:
            # calculate min and max vals for the colorbar looking at the min/max vals for both the early and late periods
            vmin = min(np.nanmin(maps[0]), np.nanmin(maps[1]))
            vmax = max(np.nanmax(maps[0]), np.nanmax(maps[1]))
            cmap = 'viridis'
        else:
            # finding max values of absolute vals (whats the biggest difference found)
            sig_limit = np.nanmax(np.abs(maps[j]))
            # if there's no change, still need some sort of color bar, so if it's 0 then make it 0.1
            if sig_limit == 0: sig_limit = 0.1 
            # making min max values by using the val found above and doing neg and pos versions
            vmin, vmax = -sig_limit, sig_limit
            cmap = 'RdBu_r'
        # 
        im = axes[j].imshow(maps[j], origin='lower', cmap=cmap, 
                            vmin=vmin, vmax=vmax, aspect='auto')
        axes[j].set_title(titles[j], fontsize=14)
        cb = fig.colorbar(im, ax=axes[j])
        cb.set_label("standardized differences from baseline vals", fontsize=10)
        axes[j].set_xlabel("longitude")
        axes[j].set_ylabel("latitude")

    plt.tight_layout()
    plt.show()   

### **running functions**

In [ ]:
# This runs in big function notebook, don't run here to save computational resources

#X_data, y_data, var_std_array = get_cnn_tensors(model_list, 'ssp119', data_path, use_anomaly=True) 
# using 2015-2024 for early and 2055-2064 for late & ssp119 for scenario 

### I think we don't need this below?

In [ ]:
# import xarray as xr 

# # save tensor prep nc file
# ds = xr.Dataset(
#     data_vars={
#         "x_data": (["sample", "variable", "lon", "lat"], X_data),
#         "y_label": (["sample", "label_dim"], y_data),
#         "yearly_val": (["variable", "lon", "lat"], yearly_val), 
#         "all_baseline_avgs": (["variable", "lon", "lat"], all_baseline_avgs)
#     }
# )

# # 3. Export straight to NetCDF
# ds.to_netcdf("/Users/sophiekim/Desktop/help/cnn_tensors_prep.nc")

In [ ]:
# import xarray as xr 

# # save tensor prep nc file
# ds = xr.Dataset(
#     data_vars={
#         "x_data": (["sample", "variable", "lon", "lat"], X_data),
#         "y_label": (["sample", "label_dim"], y_data),
#     }
# )

# # 3. Export straight to NetCDF
# ds.to_netcdf("/Users/sophiekim/Desktop/help/cnn_tensors_prep.nc")